# fMRI encoding analysis
Build encoding models linking unit and voxel responses using non-negative ridge regression
1. Which unit type explains the most variance in each ROI?
    - Fit separate models
2. How much unique vs. shared variance does each unit type explain in each ROI?
    - Fit joint models and apply variance partitioning


In [ ]:
from myutils.nsd import (
    download_sessions, prepro_img_meta, prepro_resp,
    define_rois, extract_roi_responses, save_subj_img_ids,
)
from facebody.selectivity import print_grand_floc_dprime_summary
from facebody.encoding import (
    EncodingPipeline, EncodingConfig,
    stats_pairwise_comps_sep,
)
from facebody.plotting import (
    plot_encoding_sep, plot_encoding_sep_layers,
    plot_encoding_varpart, plot_encoding_varpart_layers,
    plot_varpart_integrated,
)

device = "cuda:0"

In [ ]:
# Specify model, layers, subjects, and ROIs
model_name = "alexnet_ecoset"
layers = [f"conv{i}" for i in range(1, 6)] + [f"fc{i}" for i in range(6, 8)]
print_grand_floc_dprime_summary(
    model_name, layers,
    controlled=True, method="layer-mean",
)

subjects = [f"subj{i:02d}" for i in range(1, 9)]
rois = [
    "OFA", "FFA", "aTL-faces",
    "EBA", "FBA", "mTL-bodies",
    "FFA&FBA",
]

## Preprocess fMRI data
- Natural Scenes Dataset (NSD; Allen et al., 2022)
    - N=8 participants
    - Face-selective, body-selective, and overlap ROIs defined using same fLoc

In [ ]:
# Download and preprocess fMRI data
download_sessions(subjects)
prepro_img_meta()
prepro_resp(subjects)
define_rois(subjects)
extract_roi_responses(subjects)
save_subj_img_ids(subjects)

## Fit and evaluate encoding model

In [ ]:
# Run encoding analysis
cfg = EncodingConfig(
    model_name=model_name,
    layers=layers,
    controlled=True,
    subjects=subjects,
    rois=rois,
    device=device,
    use_positive=True,
)
en = EncodingPipeline(cfg)
en.ensure_activs(save_to_disk=True)
en.ensure_nmf(force=False)
en.tune_alpha(force=True)
res = en.run(analyses=("sep", "fb_varpart", "delta_m"))

## Prediction performance

In [ ]:
# Fig. 3B: Explained variance per unit type
layer = "fc7"
plot_encoding_sep(
    model_name, layer, rois, r2_adj=True,
    fig_title=f"AlexNet-Sup. (Ecoset)\n{layer}", out_path="main/encoding_sep.pdf",
)
stats_pairwise_comps_sep(model_name, layer, rois=["OFA", "FFA", "aTL-faces"])
stats_pairwise_comps_sep(model_name, layer, rois=["EBA", "FBA", "mTL-bodies"])
stats_pairwise_comps_sep(model_name, layer, rois=["FFA&FBA"])

In [ ]:
# Fig. S5: Explained variance per unit type across layers
plot_encoding_sep_layers(
    model_name, layers, rois+["V1"], r2_adj=True,
    fig_title=f"AlexNet-Sup. (Ecoset)", out_path="suppl/encoding_sep_layers.pdf",
)

## Variance partitioning

In [ ]:
# Fig. 4A: Variance partitioning of unit types
layer = "fc7"
plot_encoding_varpart(
    model_name, layer, rois, r2_adj=True, show_delta=True,
    fig_title=f"AlexNet-Sup. (Ecoset)\n{layer}", out_path="main/encoding_varpart.pdf",
)

In [ ]:
# Fig. S6: Variance partitioning of unit types across layers
plot_encoding_varpart_layers(
    model_name, layers, rois+["V1"], r2_adj=True, show_delta=True,
    fig_title=f"AlexNet-Sup. (Ecoset)", out_path="suppl/encoding_varpart_layers.pdf",
)

In [ ]:
# Fig. 4B: Proportion of integrated variance across cortical hierarchy for both region types
plot_varpart_integrated(
    model_name="alexnet_ecoset", layers=layers, face_rois=("OFA", "FFA", "aTL-faces"),
    body_rois=("EBA", "FBA", "mTL-bodies"), r2_adj=True, include_delta=True,
    out_path="main/encoding_integrated.pdf",
)